# Wan2.1 T2V + Lynx (Identity) on Free T4 – ComfyUI Colab

This streamlined notebook keeps only the essentials:
- ComfyUI with GGUF support
- Wan2.1-T2V-14B Q4_K_M GGUF (T2V, quantized for T4)
- Kijai's WanVideoWrapper + Lynx identity controls
- Lightx2v T2V LoRA plus four user LoRA slots

Cells:
1. Prepare environment (lean installs for T4)
2. Download required models/LoRAs
3. Launch ComfyUI with a Cloudflare tunnel (stdout drained without log spam)


In [ ]:
# @markdown # 1. Prepare Environment (ComfyUI + GGUF + WanVideoWrapper + Lynx)
!nvidia-smi

import os

%cd /content

!apt-get update -y
!apt-get install -y aria2

# Core deps – pinned for Colab T4
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 triton==3.2.0 sageattention==1.0.6
# Trimmed extras (keeps video/io basics, drops SAM/Ultralytics/dual onnxruntime wheels)
!pip install -q av spandrel albumentations insightface opencv-python

# ComfyUI core
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI/custom_nodes

# GGUF loader
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI_GGUF"):
    !git clone https://github.com/Isi-dev/ComfyUI_GGUF.git

# KJ nodes (optimizations etc.)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git

# WanVideoWrapper (Lynx + Wan video)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper"):
    !git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git

# ComfyUI-Manager (optional, for managing/updating nodes)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

%cd /content/ComfyUI

# Install Python deps for custom nodes (if any extra are needed)
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-KJNodes/requirements.txt || echo "KJNodes extra reqs done or not needed."
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt || echo "WanVideoWrapper extra reqs done or not needed."

print("Environment prepared. Next: configure models and LoRAs in the next cell.")


In [ ]:
# @markdown # 2. Models and LoRAs – Configure and Download

from pathlib import Path
import os
from urllib.parse import urlparse

# @markdown ### (Optional) CivitAI API key
# @markdown If you download models/LoRAs from CivitAI, set your API key here.
civitai_api_key = ""  # @param {type:"string"}

# @markdown ### Base model + text encoder + VAE (Wan2.1 T2V 14B GGUF)
wan_gguf_url = "https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q4_K_M.gguf"  # @param {type:"string"}
text_encoder_url = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn.safetensors"  # @param {type:"string"}
vae_url = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors"  # @param {type:"string"}

# @markdown ### Lynx layers (IP + Ref + Resampler)
lynx_ip_layers_url = "https://huggingface.co/vantagewithai/Lynx-GGUF/resolve/main/lite_ip/Wan2_1-T2V-14B-Lynx_lite_ip_layers_Q8_0.gguf"  # @param {type:"string"}
lynx_ref_layers_url = "https://huggingface.co/vantagewithai/Lynx-GGUF/resolve/main/full_ref/Wan2_1-T2V-14B-Lynx_full_ref_layers_Q8_0.gguf"  # @param {type:"string"}
lynx_resampler_url = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lynx/lynx_full_resampler_fp32.safetensors"  # @param {type:"string"}
lynx_lite_ip_layers_url = lynx_ip_layers_url  # use the same GGUF IP adapter by default

# @markdown ### Lightx2v T2V LoRA (speed)
lightx2v_lora_url = "https://huggingface.co/lightx2v/Wan2.1-T2V-14B-StepDistill-CfgDistill-Lightx2v/resolve/main/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors"  # @param {type:"string"}

# @markdown ---
# @markdown ### 4 General LoRA slots (style / motion / whatever)
use_lora1 = True  # @param {type:"boolean"}
lora1_url = "https://civitai.com/api/download/models/2336470?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora2 = True  # @param {type:"boolean"}
lora2_url = "https://civitai.com/api/download/models/2021242?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora3 = True  # @param {type:"boolean"}
lora3_url = "https://civitai.com/api/download/models/1971163?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora4 = True  # @param {type:"boolean"}
lora4_url = "https://civitai.com/api/download/models/2022080?type=Model&format=SafeTensor"  # @param {type:"string"}

print("Configured URLs. Now downloading models and LoRAs...")

# Save the workflow JSON locally so you always load the intended graph
workflow_json = r"""{
  "id": "7200c272-1c07-4834-9c36-c6ec5c140c21",
  "revision": 0,
  "last_node_id": 76,
  "last_link_id": 139,
  "nodes": [
    {
      "id": 28,
      "type": "WanVideoSetBlockSwap",
      "pos": [
        1164.3349609375,
        456.0657043457031
      ],
      "size": [
        209.6841796875,
        46
      ],
      "flags": {},
      "order": 25,
      "mode": 0,
      "inputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "link": 119
        },
        {
          "name": "block_swap_args",
          "shape": 7,
          "type": "BLOCKSWAPARGS",
          "link": 36
        }
      ],
      "outputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "links": [
            35
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoSetBlockSwap"
      },
      "widgets_values": [],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 32,
      "type": "WanVideoDecode",
      "pos": [
        3053.248046875,
        43.88642501831055
      ],
      "size": [
        270,
        198
      ],
      "flags": {},
      "order": 27,
      "mode": 0,
      "inputs": [
        {
          "name": "vae",
          "type": "WANVAE",
          "link": 39
        },
        {
          "name": "samples",
          "type": "LATENT",
          "link": 40
        }
      ],
      "outputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "links": [
            109
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoDecode"
      },
      "widgets_values": [
        false,
        272,
        272,
        144,
        128,
        "default"
      ]
    },
    {
      "id": 61,
      "type": "SetNode",
      "pos": [
        532.801513671875,
        1144.129638671875
      ],
      "size": [
        210,
        60
      ],
      "flags": {
        "collapsed": true
      },
      "order": 16,
      "mode": 0,
      "inputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "link": 120
        }
      ],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            121
          ]
        }
      ],
      "title": "Set_input_image",
      "properties": {
        "previousName": "input_image"
      },
      "widgets_values": [
        "input_image"
      ],
      "color": "#2a363b",
      "bgcolor": "#3f5159"
    },
    {
      "id": 63,
      "type": "MarkdownNote",
      "pos": [
        -122.2835464477539,
        620.6234741210938
      ],
      "size": [
        434.28240966796875,
        283.2594299316406
      ],
      "flags": {},
      "order": 0,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "properties": {},
      "widgets_values": [
        "Model links:\n\n- Wan2.1-T2V-14B GGUF (Q4_K_M, CUDA): https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q4_K_M.gguf\n- Text encoder (non-scaled FP8): https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn.safetensors\n- VAE: https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors\n\nLynx (GGUF + resampler):\n- Lite IP (Q8_0): https://huggingface.co/vantagewithai/Lynx-GGUF/resolve/main/lite_ip/Wan2_1-T2V-14B-Lynx_lite_ip_layers_Q8_0.gguf\n- Full Ref (Q8_0): https://huggingface.co/vantagewithai/Lynx-GGUF/resolve/main/full_ref/Wan2_1-T2V-14B-Lynx_full_ref_layers_Q8_0.gguf\n- Resampler FP32: https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lynx/lynx_full_resampler_fp32.safetensors\n\nLoRA:\n- Lightx2v: https://huggingface.co/lightx2v/Wan2.1-T2V-14B-StepDistill-CfgDistill-Lightx2v/resolve/main/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors\n"
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 21,
      "type": "PreviewImage",
      "pos": [
        1080.525390625,
        1056.8209228515625
      ],
      "size": [
        210,
        258
      ],
      "flags": {},
      "order": 21,
      "mode": 0,
      "inputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "link": 113
        }
      ],
      "outputs": [],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.3.59",
        "Node name for S&R": "PreviewImage"
      },
      "widgets_values": []
    },
    {
      "id": 65,
      "type": "Note",
      "pos": [
        1593.0069580078125,
        1237.267822265625
      ],
      "size": [
        210,
        91.1364974975586
      ],
      "flags": {},
      "order": 1,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "properties": {},
      "widgets_values": [
        "In the original code, the prompt for the reference extraction pass was hardcoded as this."
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 60,
      "type": "WanVideoSetLoRAs",
      "pos": [
        1203.847412109375,
        598.7035522460938
      ],
      "size": [
        178.5533203125,
        46
      ],
      "flags": {},
      "order": 23,
      "mode": 0,
      "inputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "link": 117
        },
        {
          "name": "lora",
          "shape": 7,
          "type": "WANVIDLORA",
          "link": 139
        }
      ],
      "outputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "links": [
            119
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "174cba575912a534e8c106c0c318c48f3df0e053",
        "Node name for S&R": "WanVideoSetLoRAs"
      },
      "widgets_values": [],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 64,
      "type": "Note",
      "pos": [
        730.4488525390625,
        1214.236328125
      ],
      "size": [
        320.8675842285156,
        233.28762817382812
      ],
      "flags": {},
      "order": 2,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "title": "Note: Insightface",
      "properties": {},
      "widgets_values": [
        "Original code and this node uses insightface only to crop the image, you can use other crop automatic face croppers or manually crop the face for the ip_image as well. The image is tiny 112x112\n\nWhile InsightFace code is MIT licensed, the Buffalo -model used has strictly \"non-commercial research purposes only\" -license\n\nFor the actual encoding Arcface model is used, I couldn't find clear answer to what it's license is.\n\nFor the reference the original code uses different slightly more zoomed out cropping  for 256x256 input image, you can do this manually as well."
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 36,
      "type": "PreviewImage",
      "pos": [
        1324.6285400390625,
        1061.2957763671875
      ],
      "size": [
        210,
        258
      ],
      "flags": {},
      "order": 22,
      "mode": 0,
      "inputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "link": 114
        }
      ],
      "outputs": [],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.3.59",
        "Node name for S&R": "PreviewImage"
      },
      "widgets_values": []
    },
    {
      "id": 66,
      "type": "Note",
      "pos": [
        1838.8494873046875,
        434.2574768066406
      ],
      "size": [
        351.72918701171875,
        174.49427795410156
      ],
      "flags": {},
      "order": 3,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "properties": {},
      "widgets_values": [
        "lite lynx model only includes ip adapter for face id.\n\nfull lynx model also includes reference adapter, which works by first running your reference image through the model for 1 step, and the main inference then uses those extracted values on each step.\n\nThe reference adapter feels extremely strong and usually reducing it's strength or limiting the blocks it is applied to helps avoid that"
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 52,
      "type": "LynxEncodeFaceIP",
      "pos": [
        1087.304931640625,
        905.4612426757812
      ],
      "size": [
        206.337890625,
        46
      ],
      "flags": {},
      "order": 20,
      "mode": 0,
      "inputs": [
        {
          "name": "resampler",
          "type": "LYNXRESAMPLER",
          "link": 89
        },
        {
          "name": "ip_image",
          "type": "IMAGE",
          "link": 95
        }
      ],
      "outputs": [
        {
          "name": "lynx_face_embeds",
          "type": "LYNXIP",
          "links": [
            126
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "174cba575912a534e8c106c0c318c48f3df0e053",
        "Node name for S&R": "LynxEncodeFaceIP"
      },
      "widgets_values": []
    },
    {
      "id": 53,
      "type": "LynxInsightFaceCrop",
      "pos": [
        767.8928833007812,
        1115.97900390625
      ],
      "size": [
        186.9546875,
        46
      ],
      "flags": {},
      "order": 18,
      "mode": 0,
      "inputs": [
        {
          "name": "image",
          "type": "IMAGE",
          "link": 121
        }
      ],
      "outputs": [
        {
          "name": "ip_image",
          "type": "IMAGE",
          "links": [
            95,
            113
          ]
        },
        {
          "name": "ref_image",
          "type": "IMAGE",
          "links": [
            114,
            137
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "174cba575912a534e8c106c0c318c48f3df0e053",
        "Node name for S&R": "LynxInsightFaceCrop"
      },
      "widgets_values": []
    },
    {
      "id": 57,
      "type": "ImageConcatMulti",
      "pos": [
        3052.36669921875,
        315.6687316894531
      ],
      "size": [
        270,
        150
      ],
      "flags": {},
      "order": 28,
      "mode": 0,
      "inputs": [
        {
          "name": "image_1",
          "type": "IMAGE",
          "link": 109
        },
        {
          "name": "image_2",
          "shape": 7,
          "type": "IMAGE",
          "link": 122
        }
      ],
      "outputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "links": [
            112
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfyui-kjnodes",
        "ver": "b1fffd33ba8b14cb4900e2c557f4a2ba0b340ff5"
      },
      "widgets_values": [
        2,
        "left",
        true,
        null
      ]
    },
    {
      "id": 62,
      "type": "GetNode",
      "pos": [
        3057.438720703125,
        516.5675659179688
      ],
      "size": [
        210,
        60
      ],
      "flags": {
        "collapsed": true
      },
      "order": 4,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            122
          ]
        }
      ],
      "title": "Get_input_image",
      "properties": {},
      "widgets_values": [
        "input_image"
      ],
      "color": "#2a363b",
      "bgcolor": "#3f5159"
    },
    {
      "id": 72,
      "type": "Note",
      "pos": [
        -626.7315673828125,
        664.1754760742188
      ],
      "size": [
        370.65289306640625,
        129.35208129882812
      ],
      "flags": {},
      "order": 5,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "properties": {},
      "widgets_values": [
        "Original implementation uses full ip layers with full ref layers, I don't know if there's some mistake in my implementation as the full ip adapter seems very weak, and using lite ip instead seems better, and also uses less memory."
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 29,
      "type": "WanVideoBlockSwap",
      "pos": [
        1147.027587890625,
        164.4652099609375
      ],
      "size": [
        281.404296875,
        202
      ],
      "flags": {},
      "order": 6,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "block_swap_args",
          "type": "BLOCKSWAPARGS",
          "links": [
            36
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoBlockSwap"
      },
      "widgets_values": [
        35,
        false,
        false,
        true,
        0,
        1,
        false
      ],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 55,
      "type": "WanVideoAddLynxEmbeds",
      "pos": [
        1840.908447265625,
        668.7803955078125
      ],
      "size": [
        281.57501220703125,
        254
      ],
      "flags": {},
      "order": 24,
      "mode": 0,
      "inputs": [
        {
          "name": "embeds",
          "type": "WANVIDIMAGE_EMBEDS",
          "link": 101
        },
        {
          "name": "vae",
          "shape": 7,
          "type": "WANVAE",
          "link": 107
        },
        {
          "name": "lynx_ip_embeds",
          "shape": 7,
          "type": "LYNXIP",
          "link": 126
        },
        {
          "name": "ref_image",
          "shape": 7,
          "type": "IMAGE",
          "link": 137
        },
        {
          "name": "ref_text_embed",
          "shape": 7,
          "type": "WANVIDEOTEXTEMBEDS",
          "link": 136
        },
        {
          "name": "ref_blocks_to_use",
          "shape": 7,
          "type": "STRING",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "image_embeds",
          "type": "WANVIDIMAGE_EMBEDS",
          "links": [
            104
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "174cba575912a534e8c106c0c318c48f3df0e053",
        "Node name for S&R": "WanVideoAddLynxEmbeds"
      },
      "widgets_values": [
        0.7,
        0.6,
        2,
        0,
        1
      ]
    },
    {
      "id": 74,
      "type": "WanVideoLoraSelectMulti",
      "pos": [
        523.8887939453125,
        18.619842529296875
      ],
      "size": [
        522.9938354492188,
        342
      ],
      "flags": {},
      "order": 7,
      "mode": 0,
      "inputs": [
        {
          "name": "prev_lora",
          "shape": 7,
          "type": "WANVIDLORA",
          "link": null
        },
        {
          "name": "blocks",
          "shape": 7,
          "type": "SELECTEDBLOCKS",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "lora",
          "type": "WANVIDLORA",
          "links": [
            139
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "174cba575912a534e8c106c0c318c48f3df0e053",
        "Node name for S&R": "WanVideoLoraSelectMulti"
      },
      "widgets_values": [
        "Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors",
        1,
        "none",
        0.5,
        "none",
        1,
        "none",
        1,
        "none",
        1,
        false,
        false
      ],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 22,
      "type": "WanVideoSampler",
      "pos": [
        2550.7099609375,
        230.54959106445312
      ],
      "size": [
        327.80859375,
        881.5819091796875
      ],
      "flags": {},
      "order": 26,
      "mode": 0,
      "inputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "link": 35
        },
        {
          "name": "image_embeds",
          "type": "WANVIDIMAGE_EMBEDS",
          "link": 104
        },
        {
          "name": "text_embeds",
          "shape": 7,
          "type": "WANVIDEOTEXTEMBEDS",
          "link": 31
        },
        {
          "name": "samples",
          "shape": 7,
          "type": "LATENT",
          "link": null
        },
        {
          "name": "feta_args",
          "shape": 7,
          "type": "FETAARGS",
          "link": null
        },
        {
          "name": "context_options",
          "shape": 7,
          "type": "WANVIDCONTEXT",
          "link": null
        },
        {
          "name": "cache_args",
          "shape": 7,
          "type": "CACHEARGS",
          "link": null
        },
        {
          "name": "flowedit_args",
          "shape": 7,
          "type": "FLOWEDITARGS",
          "link": null
        },
        {
          "name": "slg_args",
          "shape": 7,
          "type": "SLGARGS",
          "link": null
        },
        {
          "name": "loop_args",
          "shape": 7,
          "type": "LOOPARGS",
          "link": null
        },
        {
          "name": "experimental_args",
          "shape": 7,
          "type": "EXPERIMENTALARGS",
          "link": null
        },
        {
          "name": "sigmas",
          "shape": 7,
          "type": "SIGMAS",
          "link": null
        },
        {
          "name": "unianimate_poses",
          "shape": 7,
          "type": "UNIANIMATE_POSE",
          "link": null
        },
        {
          "name": "fantasytalking_embeds",
          "shape": 7,
          "type": "FANTASYTALKING_EMBEDS",
          "link": null
        },
        {
          "name": "uni3c_embeds",
          "shape": 7,
          "type": "UNI3C_EMBEDS",
          "link": null
        },
        {
          "name": "multitalk_embeds",
          "shape": 7,
          "type": "MULTITALK_EMBEDS",
          "link": null
        },
        {
          "name": "freeinit_args",
          "shape": 7,
          "type": "FREEINITARGS",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "samples",
          "type": "LATENT",
          "links": [
            40
          ]
        },
        {
          "name": "denoised_samples",
          "type": "LATENT",
          "links": null
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoSampler"
      },
      "widgets_values": [
        4,
        1,
        8,
        42,
        "fixed",
        true,
        "euler/beta",
        0,
        1,
        false,
        "comfy",
        0,
        -1,
        false
      ]
    },
    {
      "id": 24,
      "type": "WanVideoEmptyEmbeds",
      "pos": [
        1491.357666015625,
        667.6817626953125
      ],
      "size": [
        272.431640625,
        126
      ],
      "flags": {},
      "order": 8,
      "mode": 0,
      "inputs": [
        {
          "name": "control_embeds",
          "shape": 7,
          "type": "WANVIDIMAGE_EMBEDS",
          "link": null
        },
        {
          "name": "extra_latents",
          "shape": 7,
          "type": "LATENT",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "image_embeds",
          "type": "WANVIDIMAGE_EMBEDS",
          "links": [
            101
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoEmptyEmbeds"
      },
      "widgets_values": [
        480,
        832,
        81
      ]
    },
    {
      "id": 58,
      "type": "VHS_VideoCombine",
      "pos": [
        3372.08154296875,
        46.2856559753418
      ],
      "size": [
        1784.03125,
        334
      ],
      "flags": {},
      "order": 29,
      "mode": 0,
      "inputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "link": 112
        },
        {
          "name": "audio",
          "shape": 7,
          "type": "AUDIO",
          "link": null
        },
        {
          "name": "meta_batch",
          "shape": 7,
          "type": "VHS_BatchManager",
          "link": null
        },
        {
          "name": "vae",
          "shape": 7,
          "type": "VAE",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "Filenames",
          "type": "VHS_FILENAMES",
          "links": null
        }
      ],
      "properties": {
        "cnr_id": "comfyui-videohelpersuite",
        "ver": "0edce8ef7ce173ac97a3ed3d6f4636029d1a4530",
        "Node name for S&R": "VHS_VideoCombine"
      },
      "widgets_values": {
        "frame_rate": 16,
        "loop_count": 0,
        "filename_prefix": "Wanlynx",
        "format": "video/h264-mp4",
        "pix_fmt": "yuv420p",
        "crf": 19,
        "save_metadata": true,
        "trim_to_audio": false,
        "pingpong": false,
        "save_output": false,
        "videopreview": {
          "hidden": false,
          "paused": false,
          "params": {
            "filename": "Wanlynx_00041.mp4",
            "subfolder": "",
            "type": "temp",
            "format": "video/h264-mp4",
            "frame_rate": 24,
            "workflow": "Wanlynx_00041.png",
            "fullpath": "N:\\AI\\ComfyUI\\temp\\Wanlynx_00041.mp4"
          }
        }
      }
    },
    {
      "id": 35,
      "type": "LoadImage",
      "pos": [
        62.843509674072266,
        1122.969482421875
      ],
      "size": [
        335.2794494628906,
        475.0510559082031
      ],
      "flags": {},
      "order": 9,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            120
          ]
        },
        {
          "name": "MASK",
          "type": "MASK",
          "links": null
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.3.59",
        "Node name for S&R": "LoadImage"
      },
      "widgets_values": [
        "20251202_015416 jpeg.jpg",
        "image"
      ]
    },
    {
      "id": 16,
      "type": "LoadLynxResampler",
      "pos": [
        62.8701057434082,
        977.65185546875
      ],
      "size": [
        350,
        82
      ],
      "flags": {},
      "order": 10,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "resampler",
          "type": "LYNXRESAMPLER",
          "links": [
            89
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "LoadLynxResampler"
      },
      "widgets_values": [
        "lynx_full_resampler_fp32.safetensors",
        "fp16"
      ]
    },
    {
      "id": 13,
      "type": "WanVideoExtraModelSelect",
      "pos": [
        -671.5703735351562,
        535.4902954101562
      ],
      "size": [
        477.6412658691406,
        59.95210266113281
      ],
      "flags": {},
      "order": 17,
      "mode": 0,
      "inputs": [
        {
          "name": "prev_model",
          "shape": 7,
          "type": "VACEPATH",
          "link": 134
        }
      ],
      "outputs": [
        {
          "name": "extra_model",
          "type": "VACEPATH",
          "links": [
            135
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoExtraModelSelect"
      },
      "widgets_values": [
        "Wan2_1-T2V-14B-Lynx_full_ref_layers_Q8_0.gguf"
      ],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 69,
      "type": "WanVideoExtraModelSelect",
      "pos": [
        -683.1736450195312,
        419.47369384765625
      ],
      "size": [
        477.6412658691406,
        59.95210266113281
      ],
      "flags": {},
      "order": 11,
      "mode": 0,
      "inputs": [
        {
          "name": "prev_model",
          "shape": 7,
          "type": "VACEPATH",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "extra_model",
          "type": "VACEPATH",
          "links": [
            134
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoExtraModelSelect"
      },
      "widgets_values": [
        "Wan2_1-T2V-14B-Lynx_lite_ip_layers_Q8_0.gguf"
      ],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 31,
      "type": "WanVideoVAELoader",
      "pos": [
        2551.93994140625,
        30.22148323059082
      ],
      "size": [
        334.9571533203125,
        106
      ],
      "flags": {},
      "order": 12,
      "mode": 0,
      "inputs": [
        {
          "name": "compile_args",
          "shape": 7,
          "type": "WANCOMPILEARGS",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "vae",
          "type": "WANVAE",
          "links": [
            39,
            107
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoVAELoader"
      },
      "widgets_values": [
        "wan_2.1_vae.safetensors",
        "bf16",
        true
      ],
      "color": "#322",
      "bgcolor": "#533"
    },
    {
      "id": 51,
      "type": "WanVideoTextEncodeCached",
      "pos": [
        1852.8812255859375,
        973.4616088867188
      ],
      "size": [
        532.5263061523438,
        354.6100769042969
      ],
      "flags": {},
      "order": 13,
      "mode": 0,
      "inputs": [
        {
          "name": "extender_args",
          "shape": 7,
          "type": "WANVIDEOPROMPTEXTENDER_ARGS",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "text_embeds",
          "type": "WANVIDEOTEXTEMBEDS",
          "links": [
            136
          ]
        },
        {
          "name": "negative_text_embeds",
          "type": "WANVIDEOTEXTEMBEDS",
          "links": null
        },
        {
          "name": "positive_prompt",
          "type": "STRING",
          "links": null
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoTextEncodeCached"
      },
      "widgets_values": [
        "umt5_xxl_fp8_e4m3fn.safetensors",
        "bf16",
        "image of a face",
        "",
        "disabled",
        true,
        "gpu"
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 26,
      "type": "WanVideoTextEncodeCached",
      "pos": [
        1806.466796875,
        -51.649444580078125
      ],
      "size": [
        558.2944946289062,
        382.5255432128906
      ],
      "flags": {},
      "order": 14,
      "mode": 0,
      "inputs": [
        {
          "name": "extender_args",
          "shape": 7,
          "type": "WANVIDEOPROMPTEXTENDER_ARGS",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "text_embeds",
          "type": "WANVIDEOTEXTEMBEDS",
          "links": [
            31
          ]
        },
        {
          "name": "negative_text_embeds",
          "type": "WANVIDEOTEXTEMBEDS",
          "links": null
        },
        {
          "name": "positive_prompt",
          "type": "STRING",
          "links": null
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoTextEncodeCached"
      },
      "widgets_values": [
        "umt5_xxl_fp8_e4m3fn.safetensors",
        "bf16",
        "a beautiful white woman with black hair, thin body, and huge natural breasts is laying in bed seductively while looking directly at viewer.",
        "\u8272\u8c03\u8273\u4e3d\uff0c\u8fc7\u66dd\uff0c\u9759\u6001\uff0c\u7ec6\u8282\u6a21\u7cca\u4e0d\u6e05\uff0c\u5b57\u5e55\uff0c\u98ce\u683c\uff0c\u4f5c\u54c1\uff0c\u753b\u4f5c\uff0c\u753b\u9762\uff0c\u9759\u6b62\uff0c\u6574\u4f53\u53d1\u7070\uff0c\u6700\u5dee\u8d28\u91cf\uff0c\u4f4e\u8d28\u91cf\uff0cJPEG\u538b\u7f29\u6b8b\u7559\uff0c\u4e11\u964b\u7684\uff0c\u6b8b\u7f3a\u7684\uff0c\u591a\u4f59\u7684\u624b\u6307\uff0c\u753b\u5f97\u4e0d\u597d\u7684\u624b\u90e8\uff0c\u753b\u5f97\u4e0d\u597d\u7684\u8138\u90e8\uff0c\u7578\u5f62\u7684\uff0c\u6bc1\u5bb9\u7684\uff0c\u5f62\u6001\u7578\u5f62\u7684\u80a2\u4f53\uff0c\u624b\u6307\u878d\u5408\uff0c\u9759\u6b62\u4e0d\u52a8\u7684\u753b\u9762\uff0c\u6742\u4e71\u7684\u80cc\u666f\uff0c\u4e09\u6761\u817f\uff0c\u80cc\u666f\u4eba\u5f88\u591a\uff0c\u5012\u7740\u8d70",
        "disabled",
        true,
        "gpu"
      ],
      "color": "#432",
      "bgcolor": "#653"
    },
    {
      "id": 12,
      "type": "WanVideoModelLoader",
      "pos": [
        384.0904235839844,
        437.55438232421875
      ],
      "size": [
        609.2658081054688,
        318
      ],
      "flags": {},
      "order": 19,
      "mode": 0,
      "inputs": [
        {
          "name": "compile_args",
          "shape": 7,
          "type": "WANCOMPILEARGS",
          "link": 111
        },
        {
          "name": "block_swap_args",
          "shape": 7,
          "type": "BLOCKSWAPARGS",
          "link": null
        },
        {
          "name": "lora",
          "shape": 7,
          "type": "WANVIDLORA",
          "link": null
        },
        {
          "name": "vram_management_args",
          "shape": 7,
          "type": "VRAM_MANAGEMENTARGS",
          "link": null
        },
        {
          "name": "extra_model",
          "shape": 7,
          "type": "VACEPATH",
          "link": 135
        },
        {
          "name": "fantasytalking_model",
          "shape": 7,
          "type": "FANTASYTALKINGMODEL",
          "link": null
        },
        {
          "name": "multitalk_model",
          "shape": 7,
          "type": "MULTITALKMODEL",
          "link": null
        },
        {
          "name": "fantasyportrait_model",
          "shape": 7,
          "type": "FANTASYPORTRAITMODEL",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "model",
          "type": "WANVIDEOMODEL",
          "links": [
            117
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoModelLoader"
      },
      "widgets_values": [
        "wan2.1-t2v-14b-Q4_K_M.gguf",
        "fp16",
        "disabled",
        "cuda",
        "sageattn",
        "default"
      ],
      "color": "#223",
      "bgcolor": "#335"
    },
    {
      "id": 37,
      "type": "WanVideoTorchCompileSettings",
      "pos": [
        -42.34662628173828,
        193.6160888671875
      ],
      "size": [
        342.74609375,
        250
      ],
      "flags": {},
      "order": 15,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "torch_compile_args",
          "type": "WANCOMPILEARGS",
          "links": [
            111
          ]
        }
      ],
      "properties": {
        "cnr_id": "ComfyUI-WanVideoWrapper",
        "ver": "37365817e82c3d6057fcb0a5d7f952f7376f096a",
        "Node name for S&R": "WanVideoTorchCompileSettings"
      },
      "widgets_values": [
        "none",
        false,
        "default",
        false,
        64,
        false,
        128,
        false,
        false
      ],
      "color": "#223",
      "bgcolor": "#335"
    }
  ],
  "links": [
    [
      31,
      26,
      0,
      22,
      2,
      "WANVIDEOTEXTEMBEDS"
    ],
    [
      35,
      28,
      0,
      22,
      0,
      "WANVIDEOMODEL"
    ],
    [
      36,
      29,
      0,
      28,
      1,
      "BLOCKSWAPARGS"
    ],
    [
      39,
      31,
      0,
      32,
      0,
      "WANVAE"
    ],
    [
      40,
      22,
      0,
      32,
      1,
      "LATENT"
    ],
    [
      89,
      16,
      0,
      52,
      0,
      "LYNXRESAMPLER"
    ],
    [
      95,
      53,
      0,
      52,
      1,
      "IMAGE"
    ],
    [
      101,
      24,
      0,
      55,
      0,
      "WANVIDIMAGE_EMBEDS"
    ],
    [
      104,
      55,
      0,
      22,
      1,
      "WANVIDIMAGE_EMBEDS"
    ],
    [
      107,
      31,
      0,
      55,
      1,
      "WANVAE"
    ],
    [
      109,
      32,
      0,
      57,
      0,
      "IMAGE"
    ],
    [
      111,
      37,
      0,
      12,
      0,
      "WANCOMPILEARGS"
    ],
    [
      112,
      57,
      0,
      58,
      0,
      "IMAGE"
    ],
    [
      113,
      53,
      0,
      21,
      0,
      "IMAGE"
    ],
    [
      114,
      53,
      1,
      36,
      0,
      "IMAGE"
    ],
    [
      117,
      12,
      0,
      60,
      0,
      "WANVIDEOMODEL"
    ],
    [
      119,
      60,
      0,
      28,
      0,
      "WANVIDEOMODEL"
    ],
    [
      120,
      35,
      0,
      61,
      0,
      "*"
    ],
    [
      121,
      61,
      0,
      53,
      0,
      "IMAGE"
    ],
    [
      122,
      62,
      0,
      57,
      1,
      "IMAGE"
    ],
    [
      126,
      52,
      0,
      55,
      2,
      "LYNXIP"
    ],
    [
      134,
      69,
      0,
      13,
      0,
      "VACEPATH"
    ],
    [
      135,
      13,
      0,
      12,
      4,
      "VACEPATH"
    ],
    [
      136,
      51,
      0,
      55,
      4,
      "WANVIDEOTEXTEMBEDS"
    ],
    [
      137,
      53,
      1,
      55,
      3,
      "IMAGE"
    ],
    [
      139,
      74,
      0,
      60,
      1,
      "WANVIDLORA"
    ]
  ],
  "groups": [],
  "config": {},
  "extra": {
    "ds": {
      "scale": 1,
      "offset": [
        -201.0957219132572,
        20.854359923298375
      ]
    },
    "frontendVersion": "1.34.8",
    "workflowRendererVersion": "LG",
    "VHS_latentpreview": false,
    "VHS_latentpreviewrate": 0,
    "VHS_MetadataImage": true,
    "VHS_KeepIntermediate": true,
    "node_versions": {
      "ComfyUI-WanVideoWrapper": "1ba1a1662b2db34969c4dbe2d13d8a4780491d30",
      "ComfyUI-KJNodes": "9d7af919b91838fb22e31ad0107a6ddcf8bd7f3f",
      "comfy-core": "0.3.61",
      "comfyui-videohelpersuite": "8e4d79471bf1952154768e8435a9300077b534fa"
    }
  },
  "version": 0.4
}"""
workflow_path = Path("/content/ComfyUI/workflows/wanvideo_T2V_14B_lynx.json")
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(workflow_json)
print(f"Saved workflow to {workflow_path}")

headers = {}
if civitai_api_key:
    headers["Authorization"] = f"Bearer {civitai_api_key}"

def download_file(url, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = Path(urlparse(url).path).name or "download.bin"
    dest_path = dest_dir / filename

    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"Already exists: {dest_path}")
        return dest_path
    print(f"Downloading {url} -> {dest_path}")
    header_args = "".join([f" --header='{k}: {v}'" for k, v in headers.items()])
    cmd = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M{header_args} '{url}' -d '{dest_dir}' -o '{filename}'"
    rc = os.system(cmd)
    if rc != 0:
        print("aria2c failed or not installed, falling back to wget...")
        header_flags = "".join([f" --header='{k}: {v}'" for k, v in headers.items()])
        cmd2 = f"wget -O '{dest_path}'{header_flags} '{url}'"
        rc2 = os.system(cmd2)
        if rc2 != 0:
            raise RuntimeError(f"Download failed for {url}")
    return dest_path

# Base paths
diffusion_dir = "/content/ComfyUI/models/unet"
text_enc_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
lora_dir = "/content/ComfyUI/models/loras"
lynx_dir = "/content/ComfyUI/models/lynx"

# Download core Wan2.1 T2V GGUF and encoders
wan_gguf_path = download_file(wan_gguf_url, diffusion_dir)
text_enc_path = download_file(text_encoder_url, text_enc_dir)
vae_path = download_file(vae_url, vae_dir)

# Download Lynx layers
lynx_ip_path = download_file(lynx_ip_layers_url, lynx_dir)
lynx_ref_path = download_file(lynx_ref_layers_url, lynx_dir)
lynx_lite_ip_path = download_file(lynx_lite_ip_layers_url, lynx_dir)
lynx_resampler_path = download_file(lynx_resampler_url, lynx_dir)

# Download Lightx2v T2V LoRA
lightx2v_lora_path = download_file(lightx2v_lora_url, lora_dir)

# Download up to 4 general LoRAs
lora_paths = []

if use_lora1 and lora1_url:
    lora_paths.append(("lora1", download_file(lora1_url, lora_dir)))
if use_lora2 and lora2_url:
    lora_paths.append(("lora2", download_file(lora2_url, lora_dir)))
if use_lora3 and lora3_url:
    lora_paths.append(("lora3", download_file(lora3_url, lora_dir)))
if use_lora4 and lora4_url:
    lora_paths.append(("lora4", download_file(lora4_url, lora_dir)))

print("\nDownloads complete.")
print("Wan GGUF:", wan_gguf_path)
print("Text encoder:", text_enc_path)
print("VAE:", vae_path)
print("Lynx:", lynx_ip_path, lynx_ref_path, lynx_lite_ip_path, lynx_resampler_path)
print("Lightx2v LoRA:", lightx2v_lora_path)
print("User LoRAs:", lora_paths)


In [ ]:
# @markdown # 3. Launch ComfyUI (Lynx T2V Workflow)
import subprocess, threading, time, os, re
from IPython.display import HTML

%cd /content/ComfyUI

# Basic environment tweaks for Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"
os.environ["XDG_RUNTIME_DIR"] = "/tmp"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
os.environ["FFMPEG_LOGLEVEL"] = "quiet"


def start_comfy():
    cmd = ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"]
    subprocess.Popen(cmd)

# Start ComfyUI in the background
thread = threading.Thread(target=start_comfy, daemon=True)
thread.start()

# Give it a bit of time to boot
time.sleep(10)
print("ComfyUI server started on port 8188.")

public_url = None

use_cloudflared = True  # @param {type:"boolean"}

if use_cloudflared:
    print("Using cloudflared tunnel (no log spam; stdout drained in background).")

    if not os.path.exists("/usr/local/bin/cloudflared"):
        !curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared
    else:
        print("cloudflared already present, skipping download.")

    def launch_cloudflared():
        proc = subprocess.Popen(
            ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        url_holder = {"url": None}

        def drain_stdout():
            for line in iter(proc.stdout.readline, ""):
                if not line:
                    break
                if url_holder["url"] is None:
                    m = re.search(r"https://[\w.-]*trycloudflare\.com[^\s]*", line)
                    if m:
                        url_holder["url"] = m.group(0).strip()
            try:
                proc.stdout.close()
            except Exception:
                pass

        threading.Thread(target=drain_stdout, daemon=True).start()
        return proc, url_holder

    cf_proc, cf_url_holder = launch_cloudflared()

    # Wait up to ~3 minutes for the URL to appear
    start_time = time.time()
    while time.time() - start_time < 180:
        if cf_url_holder["url"]:
            public_url = cf_url_holder["url"]
            break
        if cf_proc.poll() is not None:
            break
        time.sleep(1)

    if cf_proc.poll() is not None and not public_url:
        print("cloudflared exited before providing a URL. Re-run this cell to retry.")
    elif not public_url:
        print("cloudflared is running but no URL was detected yet; it will continue draining logs in the background.")
else:
    print("Using Colab's internal proxy (no external tunnel).")
    try:
        from google.colab.output import eval_js
        public_url = eval_js("google.colab.kernel.proxyPort(8188)")
        print("Proxy URL:", public_url)
        display(HTML(f'<iframe src="{public_url}" style="width: 100%; height: 600px; border: 0;"></iframe>'))
    except Exception as e:
        print("Could not auto-create iframe / proxy URL:", e)
        print("You can still tunnel manually with cloudflared or ngrok if desired.")

if public_url:
    print(f"ComfyUI is available at: {public_url}")
else:
    print("No public URL detected yet. If cloudflared is still running, wait a few seconds and re-run this cell.")

print(
    """
Next steps (inside ComfyUI):

1. In ComfyUI, go to the file browser and load the Lynx T2V workflow:
   /content/ComfyUI/workflows/wanvideo_T2V_14B_lynx.json

2. Make sure the following models are loaded in the workflow nodes:
   - Unet Loader (GGUF):  Wan2_1-T2V-14B-Q4_K_M.gguf (device: cuda)
   - Text encoder:        umt5_xxl_fp8_e4m3fn.safetensors
   - VAE:                 wan_2.1_vae.safetensors
   - Lynx IP/Ref/Resampler (GGUF+resampler): the files in models/lynx

3. Add or confirm the Lightx2v T2V LoRA node in the graph
   and set its strength and steps appropriate for your VRAM/time budget
   (for example, ~0.8 strength and 4 steps with an LCM sampler).

4. Create or confirm up to four LoRA loader nodes chained on the UNet
   for the LoRAs you downloaded into models/loras.
   Each LoRA loader node has its own strength slider that you can tune
   directly in the ComfyUI interface.

5. In the text prompt node(s), set your prompt and negative prompt
   however you like for the current shot.

6. In the Lynx node, set ip_scale and ref_scale to control
   how strongly the identity and reference details are enforced
   (for example ~0.9 / 0.9 as a starting point).

7. Queue the prompt. The output video will be saved under:
   /content/ComfyUI/output

You can then download the video via the Colab file browser.
"""
)

# Keep the cell alive so the tunnel and server remain accessible
print("Keeping this cell alive; stop it manually when you're done using ComfyUI.")
try:
    while True:
        time.sleep(300)
except KeyboardInterrupt:
    print("Stopping keep-alive loop. Cloudflared/ComfyUI will stop when the processes or runtime end.")


## Final checklist for Colab T4 runs
- If CivitAI downloads are gated, paste your **civitai_api_key** in Cell 2 before running downloads.
- If any Hugging Face links require auth, set **HF_TOKEN** in the runtime environment (e.g., in Cell 1) so model pulls succeed.
- Keep the default 272x272 resolution and modest steps/frame counts for the GGUF UNet; raise cautiously if VRAM allows.
- Disable unused LoRA slots in Cell 2 to save download time and RAM.
- Cloudflared is enabled by default; if your network blocks it, set `use_cloudflared = False` in Cell 3 to fall back to the Colab proxy.
- After a runtime reset, rerun Cells 1 → 2 → 3 in order to re-prepare the environment and reload models.